# TARDIS — SNCF TGV Data Cleaning & EDA

This notebook prepares the **SNCF TGV punctuality dataset** for downstream analysis and modelling. It runs a structured, step-by-step cleaning pipeline, then closes with an exploratory data analysis section.

---

## What is in this dataset?

Each row represents **one month of punctuality data for a specific TGV route**, and contains:

- The **route** — departure station, arrival station, service type (national / international)
- The **volume** — number of trains scheduled, cancelled, and delayed
- **Average delay values** at departure and at arrival (for all trains, and for late trains only)
- A **breakdown of delay causes** — infrastructure, rolling stock, weather, traffic management, station management, and passenger handling

---

## Pipeline at a glance

| Step | Section | What it does |
|------|---------|-------------|
| 1 | Imports | Load required libraries |
| 2 | Load | Read raw CSV and keep a frozen copy for delta comparisons |
| 2.1 | Drop comment columns | Remove free-text columns that carry no structured signal |
| 3 | Deduplicate | Remove exact duplicate rows |
| 4 | Missing values — overview | Map the full extent of NaN before taking any action |
| 4.1 | Drop critical NaN | Drop rows whose key identifiers cannot be recovered |
| 4.2–4.5 | Standardise types | Parse dates, convert numerics, normalise text labels |
| 4.5.1 | Station name clustering | Fuzzy deduplication of spelling variants |
| 4.6–4.7 | Recover delay NaN | Derive missing values from algebraic relationships |
| 4.8 | Remaining NaN | Document intentionally kept nulls |
| 5 | Cleaning summary | Row / column delta and retention funnel |
| 6–10 | Feature engineering | Year, month, season, delay category, derived rates |
| 11–11.3 | Fix impossible values | Negatives, count overflows, delay hierarchy violations |
| 12 | Export dataset | Write `../cleaned_dataset.csv` |
| 13 | Data quality audit | Completeness, distributions, outliers, validity, effectiveness |
| 14 | Export audit report | Write `../cleaning_report.csv` |
| 15 | Exploratory Data Analysis | Visualise key patterns in the cleaned data |

---

> **Run order matters.** Each step depends on the output of the previous one — execute cells from top to bottom.

### 1. Imports

| Library | Role in this notebook |
|---------|-----------------------|
| `re` | Regular expressions for text cleaning (station names) |
| `Counter` | Character frequency maps used in the manual fuzzy similarity scorer |
| `pandas` | Dataframe operations and CSV I/O |
| `matplotlib` | Base plotting engine |
| `seaborn` | Statistical visualisations (boxplots, heatmaps) |

> No third-party fuzzy-matching library is used — the similarity scorer (`tok_sort_ratio`) is implemented from scratch using only `Counter` from the Python standard library.

In [ ]:
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

> **Audit trail — report accumulator.** `report` is a flat list of dictionaries. Each entry holds one `metric / value / category / reason` tuple. Every cleaning step appends to this list; at step 14 the full list is written to `cleaning_report.csv`, giving a complete and reproducible record of every transformation applied.

In [ ]:
report = []

### 2. Data loading

The raw CSV uses `;` as a column separator (standard in French-locale exports). Two copies are created immediately after loading:

| Variable | Purpose |
|----------|---------|
| `df` | **Working copy** — modified step by step throughout the pipeline |
| `original_file` | **Frozen snapshot** — never modified; used to compute before/after deltas and populate the audit report |

Keeping an untouched original makes it possible to measure exactly how much was removed or changed at each step, without re-reading the file.

In [ ]:
df = pd.read_csv("./dataset.csv", sep=";")
original_file = df.copy()
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

> **Audit trail — raw dataset snapshot.** Before any cleaning, we record the total shape, total null count, and per-column null counts. These `initial_state` entries in the report are the baseline from which all cleaning effectiveness metrics are later calculated.

In [ ]:
initial_null_total = original_file.isnull().sum().sum()
perfect_rows = original_file.dropna().shape[0]
perfect_cols = original_file.dropna(axis=1).shape[1]
nb_rows = original_file.shape[0]
nb_cols = original_file.shape[1]
nb_slots = nb_cols * nb_rows
nan_per_col = original_file.isnull().sum()
nan_per_col = nan_per_col[nan_per_col > 0].sort_values()

report += [
    {
        "metric": "initial_rows",
        "value": original_file.shape[0],
        "category": "initial_state",
        "reason": "Raw dataset before any cleaning",
    },
    {
        "metric": "initial_columns",
        "value": original_file.shape[1],
        "category": "initial_state",
        "reason": "Raw dataset before any cleaning",
    },
    {
        "metric": "initial_null_total",
        "value": int(initial_null_total),
        "category": "initial_state",
        "reason": "Total missing values across all cells",
    },
    {
        "metric": "perfect_rows",
        "value": perfect_rows,
        "category": "initial_state",
        "reason": "Rows with zero NaN — no cleaning needed",
    },
    {
        "metric": "rows_lost_if_strict_dropna",
        "value": original_file.shape[0] - perfect_rows,
        "category": "initial_state",
        "reason": "Rows lost with a strict zero-NaN policy",
    },
    {
        "metric": "perfect_columns",
        "value": perfect_cols,
        "category": "initial_state",
        "reason": "Columns with zero NaN",
    },
    {
        "metric": "cols_lost_if_strict_dropna",
        "value": original_file.shape[1] - perfect_cols,
        "category": "initial_state",
        "reason": "Columns lost with a strict zero-NaN policy",
    },
]
for col in original_file.columns:
    report.append(
        {
            "metric": f"initial_null__{col}",
            "value": int(original_file[col].isnull().sum()),
            "category": "initial_null_per_column",
            "reason": col,
        }
    )

fig, axes = plt.subplots(1, 3, figsize=(25, 6))

axes[0].pie(
    [initial_null_total, nb_slots - initial_null_total],
    labels=["Null", "Healthy slots"],
    autopct="%1.1f%%",
)
axes[0].set_title("NaN global")

axes[1].pie(
    [perfect_rows, nb_rows - perfect_rows],
    labels=["perfects rows", "rows with NaN"],
    autopct="%1.1f%%",
)
axes[1].set_title("perfect rows")

nan_per_col.plot(kind="barh", ax=axes[2])
axes[2].set_title("NaN per collumns")

plt.tight_layout()
plt.show()

### 2.1 Dropping comment columns

Three columns end with `comments` and contain free-text justifications written manually by SNCF operators:

- `Cancellation comments`
- `Departure delay comments`
- `Arrival delay comments`

Free-text fields **cannot** be encoded numerically or used in aggregations. They are removed entirely — their content is unstructured and unusable in a tabular model.

> The columns are detected automatically by the regex pattern `comments$`, so the code is robust to future renames or new comment columns appearing in later data exports.

In [ ]:
comment_cols = df.filter(regex="comments$").columns.tolist()
df = df.drop(columns=comment_cols)
print(f"Dropped {len(comment_cols)} column(s): {comment_cols}")
print(f"Remaining: {df.shape[1]} columns")

> **Audit trail** — each dropped column is logged by name so the report clearly shows which columns were removed and why.

In [ ]:
for col in comment_cols:
    report.append(
        {
            "metric": "column_dropped",
            "value": col,
            "category": "cleaning",
            "reason": "Irrelevant data — free text comment column",
        }
    )
report.append(
    {
        "metric": "columns_dropped_total",
        "value": len(comment_cols),
        "category": "cleaning",
        "reason": "Irrelevant data — free text columns removed",
    }
)
print(f"Tracked: {len(comment_cols)} column(s) logged.")

### 3. Duplicate check

An exact duplicate is a row where **every column value** is identical to another row. Duplicates are removed before any other cleaning step because keeping them would:

- **Inflate aggregate statistics** — means, totals, and counts become artificially larger than the true values
- **Distort distributions** — repeated rows make already-common patterns appear even more dominant
- **Introduce artificial correlations** — a duplicated row boosts the relationship between all its columns simultaneously

> A row is a duplicate only if all 26 columns match exactly. Two rows for the same route and month with **different** delay values are not duplicates — they are distinct, conflicting observations and are handled separately.

In [ ]:
nb_dup = df.duplicated().sum()
if nb_dup > 0:
    df = df.drop_duplicates()
    plt.pie(
        [nb_dup, nb_rows - nb_dup], labels=["duplicated", "rows"], autopct="%1.1f%%"
    )
else:
    print("No duplicates found.")

> **Audit trail** — log the number of duplicate rows removed.

In [ ]:
report.append(
    {
        "metric": "rows_dropped_duplicates",
        "value": int(nb_dup),
        "category": "cleaning",
        "reason": "Duplicate rows — same observation counted more than once",
    }
)
print(f"Tracked: {nb_dup} row(s) logged.")

### 4. Missing values — overview

Before taking any action, we visualise the full extent of missing data across all columns. The bar chart below makes the sparsest columns immediately visible.

Three strategies are applied, chosen column by column depending on what is missing:

| Strategy | Condition | Steps |
|----------|-----------|-------|
| **Drop the row** | A column that identifies the record is null — the row cannot be placed in time or attributed to a route | 4.1 |
| **Derive the value** | The missing value can be computed exactly from other columns on the same row via an algebraic formula | 4.6 – 4.7 |
| **Leave it null** | No valid recovery exists — imputing would fabricate signal that does not exist in the original data | 4.8 |

In [ ]:
nan_counts = df.isnull().sum()
nan_counts.plot(kind="barh", figsize=(10, 6))

### 4.1 Dropping rows with missing critical identifiers

A row is dropped **only** if at least one of these five columns is null:

| Column | Why it is critical |
|--------|--------------------|
| `Date` | Without a date, the record cannot be placed in time — temporal analysis and grouping are impossible |
| `Service` | Without a service type (`NATIONAL` / `INTERNATIONAL`), the record cannot be classified by route class |
| `Departure station` | Without an origin, the route is undefined |
| `Arrival station` | Without a destination, the route is undefined |
| `Number of scheduled trains` | Without this denominator, no rate (cancellation rate, punctuality rate) can be computed |

All other null values — delay averages, percentage breakdowns, etc. — are handled in later steps and are **not** a reason to discard a row here.

In [ ]:
before = len(df)
df = df.dropna(
    subset=[
        "Date",
        "Service",
        "Departure station",
        "Arrival station",
        "Number of scheduled trains",
    ]
)
plt.pie(
    [len(df), before - len(df)], labels=["useful", "unrecoverable"], autopct="%1.1f%%"
);

> **Audit trail** — log the number of rows removed due to missing critical identifiers.

In [ ]:
crit_dropped = before - len(df)
report.append(
    {
        "metric": "rows_dropped_critical_nan",
        "value": crit_dropped,
        "category": "cleaning",
        "reason": "Unusable for analysis — missing key identifiers (CRITICAL_COLS)",
    }
)
print(f"Tracked: {crit_dropped} row(s) logged.")

### 4.2 Date parsing

The `Date` column contains string dates whose format varies across years (e.g. `"2018-01"` in older exports, `"01/2019"` in newer ones). Using `format='mixed'` lets pandas infer the correct format **row by row**, which is more robust than a single fixed pattern that would fail on any variation.

The column is cast to `datetime64[ns]`, unlocking direct temporal operations used in later steps:
- `.dt.year` / `.dt.month` — step 6, extracting year and month features
- `.dt.to_period("M")` — step 15, monthly trend analysis
- Comparison operators (`>=`, `<=`) — step 13.4, date range validity check

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format="mixed")
print(f'dtype : {df["Date"].dtype}')
print(f'Range : {df["Date"].min().date()} → {df["Date"].max().date()}')

### 4.3 Numeric conversion

All columns except the four text identifiers should be numeric. Two issues are addressed:

**1. Comma decimal separators**
Some cells use the French convention (`"6,5"`) instead of the international dot (`"6.5"`). The comma is replaced before conversion.

**2. Non-parseable strings**
Any value that still cannot be converted becomes `NaN` via `errors='coerce'`. This makes data issues visible rather than crashing silently.

> A dtype guard prevents calling `.str` on columns that are already numeric — without it, applying string methods to a float column would silently produce `NaN` for every row in that column.

In [ ]:
text_cols = ["Date", "Service", "Departure station", "Arrival station"]
numeric_cols = [c for c in df.columns if c not in text_cols]

for col in numeric_cols:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.replace(",", ".")
    df[col] = pd.to_numeric(df[col], errors="coerce")

still_object = [c for c in numeric_cols if df[c].dtype == object]
if still_object:
    print(f"WARNING — still object after conversion: {still_object}")
else:
    print(f"All {len(numeric_cols)} numeric columns successfully converted.")

### 4.4 String dtype cast

By default, pandas stores text columns as `object` dtype, which allows mixed types (`str`, `None`, `float`) to coexist in the same column. Casting to `string` dtype enforces a stricter contract:

- Nulls become `pd.NA` (a typed null) instead of `None` or `float('nan')`
- String methods (`.str.upper()`, `.str.strip()`) behave consistently and predictably
- Type errors surface early rather than propagating silently through downstream operations

In [ ]:
str_cols = ["Service", "Departure station", "Arrival station"]
df[str_cols] = df[str_cols].astype("string")
print("Dtypes after cast:")
print(df[str_cols].dtypes.to_string())

### 4.5 Label normalisation

Station and service names can appear with inconsistent casing or surrounding whitespace across rows:

- `"paris montparnasse"`, `"Paris Montparnasse"`, and `"PARIS MONTPARNASSE "` all refer to the same station
- `"national"` and `"NATIONAL"` refer to the same service type

After stripping whitespace and converting to uppercase, all variants of a name collapse into one. This ensures that `groupby` operations, `value_counts`, and joins produce correct results rather than treating the same entity as multiple distinct groups.

In [ ]:
label_cols = ["Departure station", "Arrival station", "Service"]
for col in label_cols:
    df[col] = df[col].str.strip().str.upper()

for col in label_cols:
    print(f"Unique {col}: {df[col].nunique()}")

### 4.5.1 Station name clustering — fuzzy deduplication

Label normalisation (step 4.5) cannot catch **spelling variants** of the same station — for example:

- `"BORDEAUX ST JEAN"` vs. `"BORDEAUX SAINT JEAN"`
- `"MARSEILLE ST CHARLES"` vs. `"MARSEILLE SAINT CHARLES"`

Treating these as different stations splits route statistics across two groups when they belong to one, corrupting any per-station aggregation.

This section implements a full **fuzzy deduplication pipeline** in five steps:

1. **Preprocess** all station values — normalise characters, expand abbreviations
2. **Score** every pair of unique names with a token-sort similarity function
3. **Cluster** names that score above a threshold using a Union-Find structure
4. **Canonicalise** each cluster to its most frequently observed spelling
5. **Replace** all variant spellings in the dataframe with the canonical form

**Configuration — similarity threshold and normalisation constants**

| Constant | Value | Purpose |
|----------|-------|---------|
| `STATION_THRESHOLD` | `80` | Pairs scoring ≥ 80 are treated as the same station. Decrease to merge more aggressively; increase to be stricter. |
| `CONFUSABLES` | `"1"→"I"`, `"3"→"E"`, … | Normalises characters commonly confused in hand-entered or OCR data before any comparison. |
| `ABBREV` | `"ST "` → `"SAINT "` | Expands known abbreviations so abbreviated and full forms always score as equivalent. |
| `INVALID` | `""`, `"0"`, `"NAN"`, … | Placeholder strings that are not real station names — excluded from all comparison steps. |

In [ ]:
CONFUSABLES = str.maketrans(
    {
        "1": "I",
        "3": "E",
        "4": "A",
        "5": "S",
        "8": "B",
        "@": "A",
        "'": "",
        "`": "",
        "-": "",
    }
)
ABBREV = {"ST ": "SAINT "}
INVALID = {"", "0", "NAN", "NONE", "NULL", "N/A", "NA"}
STATION_THRESHOLD = 80

**Step 1 — Detect station columns and preprocess all values**

**Automatic column detection:** a column is treated as a station column only if it:
- Contains a station-related keyword in its name (`station`, `departure`, `arrival`, …)
- Is string-typed
- Contains short labels (average length < 50 characters), low digit ratio (< 5%), and low cardinality (< 10% unique values)

**Preprocessing** transforms each raw value into a stable, comparable form:
- Strip whitespace and convert to uppercase
- Apply `CONFUSABLES` character substitutions
- Collapse repeated spaces
- Expand abbreviations via `ABBREV`

In [ ]:
keywords = ("station", "departure", "arrival", "gare", "origine", "destination")
station_columns = []

for col in df.columns:
    if not any(kw in col.lower() for kw in keywords):
        continue
    if not pd.api.types.is_string_dtype(df[col]):
        continue

    sample = df[col].dropna().astype(str)
    if sample.empty:
        continue

    avg_len = sample.str.len().mean()
    digit_ratio = sample.str.count(r"\d").sum() / max(sample.str.len().sum(), 1)
    unique_ratio = df[col].nunique() / max(len(df[col]), 1)

    if (
        avg_len < 50
        and digit_ratio < 0.05
        and unique_ratio < 0.1
        and df[col].nunique() >= 5
    ):
        station_columns.append(col)

print(f"Station-like columns detected: {station_columns}")

**Steps 2–3 — Pairwise similarity scoring and Union-Find clustering**

**Similarity scoring** uses `tok_sort_ratio`: both names are sorted token-by-token before comparison, making the score **order-independent**. `"SAINT JEAN BORDEAUX"` and `"BORDEAUX SAINT JEAN"` receive the same score as identical strings. The score ranges from 0 (no similarity) to 100 (exact match).

**Union-Find clustering** groups names whose score meets the threshold:
- Each unique name starts in its own cluster
- When two names score ≥ `STATION_THRESHOLD`, their clusters are merged
- **Path compression** keeps the structure fast even with many names to compare
- The more frequent name in each merge becomes the root, so the most common real-world spelling naturally becomes the canonical label

In [ ]:
all_values = []
for col in station_columns:
    all_values.extend(df[col].astype(str).tolist())

cleaned_of_raw = {}
valid_cleaned = []

for raw in all_values:
    cleaned = str(raw).strip().upper().translate(CONFUSABLES)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    for abbr, full in ABBREV.items():
        cleaned = cleaned.replace(abbr, full)

    cleaned_of_raw[raw] = cleaned
    if cleaned not in INVALID:
        valid_cleaned.append(cleaned)

counts = Counter(valid_cleaned)
unique = list(counts.keys())
parent = {v: v for v in unique}


def tok_sort_ratio(a, b):
    a_s = " ".join(sorted(a.split()))
    b_s = " ".join(sorted(b.split()))
    t = len(a_s) + len(b_s)
    if t == 0:
        return 100
    common = sum((Counter(a_s) & Counter(b_s)).values())
    return int(200 * common / t)


for i, a in enumerate(unique):
    for b in unique[i + 1 :]:
        if tok_sort_ratio(a, b) < STATION_THRESHOLD:
            continue

        ra = a
        while parent[ra] != ra:
            parent[ra] = parent[parent[ra]]
            ra = parent[ra]

        rb = b
        while parent[rb] != rb:
            parent[rb] = parent[parent[rb]]
            rb = parent[rb]

        if ra == rb:
            continue
        if counts[ra] >= counts[rb]:
            parent[rb] = ra
        else:
            parent[ra] = rb

**Steps 4–5 — Build canonical labels and rewrite the dataframe**

Once all clusters are formed:
1. **Path compression** is applied one final time to resolve all cluster roots
2. Each cluster gets its **canonical label** — the most frequently observed spelling across all rows
3. A lookup table maps every raw value (including singletons that were never merged) to its canonical form
4. The station columns are **overwritten** with canonical labels

After this step, all spelling variants within a cluster are unified into one consistent string throughout the dataframe.

In [ ]:
root_to_members = {}
for v in unique:
    r = v
    while parent[r] != r:
        parent[r] = parent[parent[r]]
        r = parent[r]
    root_to_members.setdefault(r, []).append(v)

canonical_of = {}
for members in root_to_members.values():
    canonical = max(members, key=lambda x: counts[x])
    for member in members:
        canonical_of[member] = canonical

mapping = {}
for raw in all_values:
    mapping[raw] = canonical_of.get(cleaned_of_raw.get(raw))

for col in station_columns:
    df[col] = df[col].astype(str).map(mapping)

**Verification — inspect all merge decisions**

Only clusters where at least two distinct spellings were merged are printed. Single-member clusters (names that were already unique) are omitted to keep the output focused on actual changes.

Each block shows the canonical label and all raw spellings it absorbed, making it easy to verify that every merge decision is correct.

> If a merge looks wrong — for example, two genuinely different stations were grouped together — **increase `STATION_THRESHOLD`** to make matching stricter, then re-run from step 4.5.1.

In [ ]:
clusters: dict = {}
for raw, canon in mapping.items():
    clusters.setdefault(canon, []).append(raw)

for canon, variants in sorted(clusters.items(), key=lambda x: str(x[0])):
    if len(variants) > 1:
        print(f"\n✦ {canon}")
        for v in sorted(variants, key=str):
            print(f"    {'  ' if v == canon else '↳ '}{v!r}")

> **Audit trail** — station name clustering statistics: total number of canonical clusters formed, number of clusters where at least one spelling variant was merged, and total number of non-canonical spellings unified.

In [ ]:
nclusters = len(root_to_members)
n_merged = sum(1 for m in root_to_members.values() if len(m) > 1)
n_variants = sum(len(m) - 1 for m in root_to_members.values() if len(m) > 1)

report += [
    {
        "metric": "stationclusters_total",
        "value": nclusters,
        "category": "station_clustering",
        "reason": "Unique canonical station names after fuzzy deduplication",
    },
    {
        "metric": "stationclusters_with_merge",
        "value": n_merged,
        "category": "station_clustering",
        "reason": "Canonical names that absorbed at least one spelling variant",
    },
    {
        "metric": "station_variants_merged",
        "value": n_variants,
        "category": "station_clustering",
        "reason": "Non-canonical spellings unified into a canonical form",
    },
    {
        "metric": "station_fuzzy_threshold",
        "value": STATION_THRESHOLD,
        "category": "station_clustering",
        "reason": "token_sort_ratio threshold used for fuzzy matching",
    },
    {
        "metric": "station_columns_processed",
        "value": len(station_columns),
        "category": "station_clustering",
        "reason": "Number of station-type columns processed",
    },
]
print(f"Clusters total    : {nclusters}")
print(f"Clusters merged   : {n_merged}")
print(f"Variants merged   : {n_variants}")
print(f"Threshold used    : {STATION_THRESHOLD}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].pie(
    [n_merged, nclusters - n_merged],
    labels=["Stations avec variantes", "Stations uniques"],
    autopct="%1.1f%%",
)
axes[0].set_title("Stations fusionnées vs uniques")

cluster_sizes = {
    canon: len(variants) for canon, variants in clusters.items() if len(variants) > 1
}

pd.Series(cluster_sizes).sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Variantes fusionnées par station canonique")
axes[1].set_xlabel("Nombre de variantes")

plt.tight_layout()
plt.show()

### 4.6 Recovering departure delay NaN — algebraic derivation

The two departure-delay columns are defined by the same underlying formula:

```
dep_all = dep_late × (n_delayed / n_scheduled)
```

Where `dep_all` is the average delay across **all** trains and `dep_late` is the average delay across **late trains only**. When one value is missing but the other and the train counts are available on the same row, the missing value can be **computed exactly** by rearranging this equation.

This is deterministic recovery — not imputation or statistical estimation.

| Missing column | Recovery formula |
|---------------|-----------------|
| `Average delay of late trains at departure` | `dep_all × (n_scheduled / n_delayed)` |
| `Average delay of all trains at departure` | `dep_late × (n_delayed / n_scheduled)` |

Recovery is applied only when the counterpart is non-null and the required count is strictly positive (no division by zero is performed).

In [ ]:
col_dep_late = "Average delay of late trains at departure"
col_dep_all = "Average delay of all trains at departure"
col_n_dep = "Number of trains delayed at departure"

mask = df[col_dep_late].isna() & df[col_dep_all].notna() & (df[col_n_dep] > 0)
dep_late_filled = int(mask.sum())
df.loc[mask, col_dep_late] = (
    df.loc[mask, col_dep_all]
    * df.loc[mask, "Number of scheduled trains"]
    / df.loc[mask, col_n_dep]
)
print(f"dep_late filled from dep_all : {dep_late_filled} row(s)")

mask = df[col_dep_all].isna() & df[col_dep_late].notna() & df[col_n_dep].notna()
dep_all_filled = int(mask.sum())
df.loc[mask, col_dep_all] = (
    df.loc[mask, col_dep_late]
    * df.loc[mask, col_n_dep]
    / df.loc[mask, "Number of scheduled trains"]
)
print(f"dep_all  filled from dep_late : {dep_all_filled} row(s)")

> **Audit trail** — log how many cells were recovered for each departure delay column.

In [ ]:
report += [
    {
        "metric": "values_filled_dep_late",
        "value": dep_late_filled,
        "category": "data_recovery",
        "reason": "Computed from average departure delay + scheduled/delayed counts",
    },
    {
        "metric": "values_filled_dep_all",
        "value": dep_all_filled,
        "category": "data_recovery",
        "reason": "Computed from late-train departure delay + delayed/scheduled counts",
    },
]
print(f"Tracked: dep_late={dep_late_filled}, dep_all={dep_all_filled}")

### 4.7 Recovering arrival delay NaN — algebraic derivation

Same deterministic recovery as step 4.6, applied to the arrival-delay columns. The same formula holds:

```
arr_all = arr_late × (n_delayed / n_scheduled)
```

| Missing column | Recovery formula |
|---------------|-----------------|
| `Average delay of late trains at arrival` | `arr_all × (n_scheduled / n_delayed)` |
| `Average delay of all trains at arrival` | `arr_late × (n_delayed / n_scheduled)` |

The same validity conditions apply: the counterpart must be non-null and the count must be strictly positive.

In [ ]:
col_arr_late = "Average delay of late trains at arrival"
col_arr_all = "Average delay of all trains at arrival"
col_n_arr = "Number of trains delayed at arrival"

mask = df[col_arr_late].isna() & df[col_arr_all].notna() & (df[col_n_arr] > 0)
arr_late_filled = int(mask.sum())
df.loc[mask, col_arr_late] = (
    df.loc[mask, col_arr_all]
    * df.loc[mask, "Number of scheduled trains"]
    / df.loc[mask, col_n_arr]
)
print(f"arr_late filled from arr_all : {arr_late_filled} row(s)")

mask = df[col_arr_all].isna() & df[col_arr_late].notna() & df[col_n_arr].notna()
arr_all_filled = int(mask.sum())
df.loc[mask, col_arr_all] = (
    df.loc[mask, col_arr_late]
    * df.loc[mask, col_n_arr]
    / df.loc[mask, "Number of scheduled trains"]
)
print(f"arr_all  filled from arr_late : {arr_all_filled} row(s)")

> **Audit trail** — log how many cells were recovered for each arrival delay column.

In [ ]:
report += [
    {
        "metric": "values_filled_arr_late",
        "value": arr_late_filled,
        "category": "data_recovery",
        "reason": "Computed from average arrival delay + scheduled/delayed counts",
    },
    {
        "metric": "values_filled_arr_all",
        "value": arr_all_filled,
        "category": "data_recovery",
        "reason": "Computed from late-train arrival delay + delayed/scheduled counts",
    },
]
print(f"Tracked: arr_late={arr_late_filled}, arr_all={arr_all_filled}")

### 4.8 Remaining NaN — intentional nulls

Any NaN still present after steps 4.6–4.7 **cannot** be derived from existing data on the same row. These values are left null intentionally:

- Imputing them with a mean or median would introduce **artificial signal** not present in the original data
- Statistical models can handle NaN natively (by masking or using appropriate estimators)
- The audit report documents exactly which columns remain sparse and by how much

The output below lists every column with remaining nulls and the count of missing values in each.

In [ ]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if not remaining.empty:
    print("Columns with remaining NaN (left as null intentionally):")
    print(remaining.to_string())
else:
    print("No remaining NaN.")

> **Audit trail** — record the total null count remaining after algebraic recovery, and the algebraic coverage rate (cells recovered / initial null total). These numbers quantify how much the deterministic derivation steps reduced overall sparsity.

In [ ]:
null_post_recovery = int(df.isnull().sum().sum())
initial_null_lookup = next(
    r["value"] for r in report if r["metric"] == "initial_null_total"
)
algebraic_total = sum(
    next(r["value"] for r in report if r["metric"] == m)
    for m in [
        "values_filled_dep_late",
        "values_filled_dep_all",
        "values_filled_arr_late",
        "values_filled_arr_all",
    ]
)
algebraic_recovery_rate = round(algebraic_total / max(initial_null_lookup, 1), 4)

report += [
    {
        "metric": "null_post_recovery",
        "value": null_post_recovery,
        "category": "recovery_summary",
        "reason": "Remaining nulls after algebraic recovery (steps 4.6–4.7)",
    },
    {
        "metric": "null_recovered_algebraic",
        "value": algebraic_total,
        "category": "recovery_summary",
        "reason": "Total cells recovered via algebraic derivation",
    },
    {
        "metric": "algebraic_recovery_rate",
        "value": algebraic_recovery_rate,
        "category": "recovery_summary",
        "reason": "Recovered / initial_null_total — algebraic coverage of initial nulls",
    },
]
print(f"Nulls post recovery   : {null_post_recovery}")
print(f"Recovered algebraic   : {algebraic_total}")
print(f"Algebraic coverage    : {algebraic_recovery_rate:.1%} of initial nulls")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    [algebraic_total, null_post_recovery],
    labels=["Recovered algebraically", "Still null (intentional)"],
    autopct="%1.1f%%",
)
ax.set_title("NaN recovery — algebraic derivation coverage")
plt.tight_layout()
plt.show()

### 5. Cleaning summary

A before-and-after comparison of the dataset shape after all row and column removal steps. No rows are added during cleaning — only removed.

The retention funnel chart in the next cell shows the row count at each removal stage, making it easy to see which step had the largest impact on data volume.

In [ ]:
print(f"Original : {original_file.shape[0]} rows, {original_file.shape[1]} columns")
print(f"Cleaned  : {df.shape[0]} rows, {df.shape[1]} columns")
print(
    f"Removed  : {original_file.shape[0] - df.shape[0]} rows, {original_file.shape[1] - df.shape[1]} columns"
)

> **Audit trail** — row retention rate and a step-by-step funnel listing the row count after each removal stage. These entries quantify how much of the original data survived the pipeline.

In [ ]:
row_retention = round(len(df) / original_file.shape[0], 4)
rows_lost_dedup = next(
    r["value"] for r in report if r["metric"] == "rows_dropped_duplicates"
)
rows_lost_crit = next(
    r["value"] for r in report if r["metric"] == "rows_dropped_critical_nan"
)
cols_after_drop = original_file.shape[1] - len(comment_cols)

report += [
    {
        "metric": "row_retention_rate",
        "value": row_retention,
        "category": "pipeline_summary",
        "reason": "Fraction of original rows kept (higher = more data preserved)",
    },
    {
        "metric": "pct_rows_lost_dedup",
        "value": round(rows_lost_dedup / original_file.shape[0], 4),
        "category": "pipeline_summary",
        "reason": "Fraction of rows lost to exact deduplication",
    },
    {
        "metric": "pct_rows_lost_critical_nan",
        "value": round(rows_lost_crit / original_file.shape[0], 4),
        "category": "pipeline_summary",
        "reason": "Fraction of rows lost to missing critical identifiers",
    },
    {
        "metric": "cols_added_feature_eng",
        "value": df.shape[1] - cols_after_drop,
        "category": "pipeline_summary",
        "reason": "Net columns added by feature engineering (year, month, season, etc.)",
    },
]

steps = [
    ("Raw dataset", original_file.shape[0]),
    ("After deduplication", original_file.shape[0] - rows_lost_dedup),
    (
        "After critical NaN drop",
        original_file.shape[0] - rows_lost_dedup - rows_lost_crit,
    ),
    ("Final (cleaned)", len(df)),
]
print(f'{"Step":<35} {"Rows":>8}  {"Retained":>9}')
print("─" * 56)
for step, n in steps:
    print(f"{step:<35} {n:>8,}  {n / original_file.shape[0]:>8.1%}")

In [ ]:
step_names = [s[0] for s in steps][::-1]
step_counts = [s[1] for s in steps][::-1]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(step_names, step_counts)
for bar, count in zip(bars, step_counts):
    pct = count / original_file.shape[0]
    ax.text(
        bar.get_width() + 30,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.1%}",
        va="center",
    )
ax.set_xlabel("Number of rows")
ax.set_title("Row retention funnel across cleaning pipeline")
plt.tight_layout()
plt.show()

### 6. Year and month extraction

Year and month are extracted from the parsed `Date` column as separate integer columns. They serve three purposes:

- **Step 7** — computing the season label
- **Step 15** — temporal trend analysis and seasonal breakdowns
- **Downstream modelling** — capturing cyclic patterns in punctuality that repeat year over year

In [ ]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
print(f'Years  : {sorted(df["year"].unique().tolist())}')
print(f'Months : {sorted(df["month"].unique().tolist())}')

### 7. Season encoding

Month integers are mapped to the four Northern-hemisphere meteorological seasons:

| Season | Months | Typical effect on punctuality |
|--------|--------|-------------------------------|
| `winter` | December, January, February | Weather disruptions — snow, ice, storms |
| `spring` | March, April, May | Generally mild, fewest disruptions |
| `summer` | June, July, August | High passenger load — crowding and connection delays |
| `autumn` | September, October, November | Leaves on tracks, early storms |

Season is an important feature for punctuality modelling because weather and load effects repeat annually and are not captured by the raw month integer alone.

In [ ]:
season_map = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "autumn",
    10: "autumn",
    11: "autumn",
}
df["season"] = df["month"].map(season_map)
print("Season distribution:")
print(df["season"].value_counts().to_string())

### 8. Delay severity category

The continuous `Average delay of all trains at arrival` column is discretised into five ordered categories using thresholds aligned with SNCF operational reporting standards:

| Category | Delay range | Interpretation |
|----------|-------------|----------------|
| `early` | < 0 min | Train arrived before the scheduled time |
| `on_time` | 0 – 5 min | Within tolerance — considered punctual |
| `slight` | 5 – 15 min | Minor delay — inconvenient but common |
| `moderate` | 15 – 30 min | Significant delay — may cause missed connections |
| `severe` | > 30 min | Major disruption — triggers passenger compensation under EU Regulation 1371/2007 |

This categorical column is used as a label for classification tasks and as a grouping variable in the EDA section (step 15).

In [ ]:
df["delay_category"] = pd.cut(
    df["Average delay of all trains at arrival"],
    bins=[float("-inf"), 0, 5, 15, 30, float("inf")],
    labels=["early", "on_time", "slight", "moderate", "severe"],
)
print("Delay category distribution:")
print(df["delay_category"].value_counts().sort_index().to_string())

### 9. Cancellation rate

The **cancellation rate** is the proportion of scheduled trains that were cancelled on a given route and month:

```
cancellation_rate = Number of cancelled trains / Number of scheduled trains
```

- Ranges from `0.0` (no cancellations) to `1.0` (every scheduled train was cancelled)
- Rows where `Number of scheduled trains` is **zero** receive `NaN` instead of attempting the division — dividing by zero would produce `inf`, which corrupts all downstream statistics that use this column

In [ ]:
df["cancellation_rate"] = df["Number of cancelled trains"] / df[
    "Number of scheduled trains"
].replace(0, float("nan"))
print(
    f'cancellation_rate — min: {df["cancellation_rate"].min():.4f}  mean: {df["cancellation_rate"].mean():.4f}  max: {df["cancellation_rate"].max():.4f}'
)

### 10. Punctuality rate

The **punctuality rate** is the proportion of scheduled trains that arrived without being counted as delayed:

```
punctuality_rate = 1 − (Number of trains delayed at arrival / Number of scheduled trains)
```

- Ranges from `1.0` (no train was delayed) to `0.0` (every train was delayed)
- It is the complement of the delay rate — a higher value means better on-time performance
- Rows where `Number of scheduled trains` is **zero** receive `NaN` for the same reason as the cancellation rate

In [ ]:
df["punctuality_rate"] = 1 - (
    df["Number of trains delayed at arrival"] / df["Number of scheduled trains"]
)
print(
    f'punctuality_rate — min: {df["punctuality_rate"].min():.4f}  mean: {df["punctuality_rate"].mean():.4f}  max: {df["punctuality_rate"].max():.4f}'
)

### 11. Negative value correction

Train count columns — such as `Number of cancelled trains` or `Number of trains delayed at arrival` — represent physical counts that **cannot be negative**. A negative value is a data entry error.

**Correction strategy:** negative values are replaced with the **median of valid (≥ 0) values for the same route** (`Departure station` × `Arrival station`). Using a route-level median:
- Preserves the local distribution of the specific route, rather than pulling toward a global average
- Is robust to outliers (a route with one very high legitimate value won't inflate the replacement)
- Requires no information outside the existing dataset

Columns checked: all seven count columns — scheduled, cancelled, delayed at departure, delayed at arrival, delayed > 15 min, > 30 min, > 60 min.

In [ ]:
count_cols = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]
route_cols = ["Departure station", "Arrival station"]

neg_fixed_total = 0
for col in count_cols:
    mask = df[col] < 0
    if mask.any():
        route_median = df.groupby(route_cols)[col].transform(
            lambda x: x[x >= 0].median()
        )
        df.loc[mask, col] = route_median[mask]
        neg_fixed_total += int(mask.sum())
        print(f"[FIXED] {col}: {mask.sum()} negative value(s) → route median")
print(f"Total: {neg_fixed_total} fix(es)")

> **Audit trail** — log the total number of negative count values corrected across all columns.

In [ ]:
report.append(
    {
        "metric": "values_fixed_negative",
        "value": neg_fixed_total,
        "category": "data_correction",
        "reason": "Impossible negative count values — replaced by route-level median",
    }
)
print(f"Tracked: {neg_fixed_total} fix(es) logged.")

### 11.1 Consistency — counts cannot exceed scheduled trains

A cancelled, delayed-at-departure, or delayed-at-arrival count that is **larger than the total number of scheduled trains** on the same row is logically impossible: you cannot cancel or delay more trains than were planned.

Such violations are corrected with the same route-level median strategy as step 11 — replacing the offending value with the median of valid entries (those that do not exceed the scheduled count) for the same route.

In [ ]:
counts_vs_scheduled = [
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
]

cons_count_fixed = 0
for col in counts_vs_scheduled:
    mask = df[col].notna() & (df[col] > df["Number of scheduled trains"])
    if mask.any():
        route_median = df.groupby(route_cols)[col].transform(
            lambda x, c=col: x[
                x <= df.loc[x.index, "Number of scheduled trains"]
            ].median()
        )
        df.loc[mask, col] = route_median[mask]
        cons_count_fixed += int(mask.sum())
        print(f"[FIXED] {col}: {mask.sum()} value(s) exceeded scheduled → route median")
print(f"Total: {cons_count_fixed} fix(es)")

> **Audit trail** — log the number of values corrected for exceeding the scheduled train count.

In [ ]:
report.append(
    {
        "metric": "values_fixed_count_vs_scheduled",
        "value": cons_count_fixed,
        "category": "data_correction",
        "reason": "Counts exceeding scheduled trains — logically impossible",
    }
)
print(f"Tracked: {cons_count_fixed} fix(es) logged.")

### 11.2 Consistency — delay threshold hierarchy

By definition, every train delayed by more than 30 minutes was also delayed by more than 15 minutes. The three delay-count columns must therefore satisfy this ordering on every row:

```
> 15 min  ≥  > 30 min  ≥  > 60 min
```

Violations of this hierarchy are corrected **bottom-up** (starting from the highest threshold, working toward the lowest), replacing each violated value with the route-level median of valid values for that column and route — the same strategy as steps 11 and 11.1.

In [ ]:
delay_hierarchy = [
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]

hier_fixed = 0
for i in range(len(delay_hierarchy) - 1):
    col_higher = delay_hierarchy[i + 1]
    col_lower = delay_hierarchy[i]
    mask = (
        df[col_higher].notna()
        & df[col_lower].notna()
        & (df[col_higher] > df[col_lower])
    )
    if mask.any():
        route_median = df.groupby(route_cols)[col_higher].transform(
            lambda x, cl=col_lower: x[x <= df.loc[x.index, cl]].median()
        )
        df.loc[mask, col_higher] = route_median[mask]
        hier_fixed += int(mask.sum())
        print(f"[FIXED] {col_higher} > {col_lower}: {mask.sum()} row(s)")
print(f"Total: {hier_fixed} fix(es)")

> **Audit trail** — log the number of delay hierarchy violations corrected.

In [ ]:
report.append(
    {
        "metric": "values_fixed_delay_hierarchy",
        "value": hier_fixed,
        "category": "data_correction",
        "reason": "Delay hierarchy violation — higher threshold exceeded lower threshold",
    }
)
print(f"Tracked: {hier_fixed} fix(es) logged.")

### 11.3 Recomputing derived rates after corrections

`cancellation_rate` and `punctuality_rate` were first computed in steps 9–10 using the **original, uncorrected** count columns. Steps 11, 11.1, and 11.2 have since fixed negative values, impossible counts, and hierarchy violations in those source columns.

The two rates are recomputed here from the corrected columns to ensure they are consistent with the cleaned data. Rows where `Number of scheduled trains` is zero still receive `NaN` to prevent division by zero.

In [ ]:
df["cancellation_rate"] = df["Number of cancelled trains"] / df[
    "Number of scheduled trains"
].replace(0, float("nan"))
df["punctuality_rate"] = 1 - (
    df["Number of trains delayed at arrival"]
    / df["Number of scheduled trains"].replace(0, float("nan"))
)

print(
    f'cancellation_rate — min: {df["cancellation_rate"].min():.4f}  mean: {df["cancellation_rate"].mean():.4f}  max: {df["cancellation_rate"].max():.4f}'
)
print(
    f'punctuality_rate  — min: {df["punctuality_rate"].min():.4f}  mean: {df["punctuality_rate"].mean():.4f}  max: {df["punctuality_rate"].max():.4f}'
)

In [ ]:
labels = ["Negative values", "Count > scheduled", "Hierarchy violations"]
values = [neg_fixed_total, cons_count_fixed, hier_fixed]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, str(val), ha="center"
    )
ax.set_ylabel("Number of corrections")
ax.set_title("Impossible values corrected per type")
plt.tight_layout()
plt.show()

### 12. Export — cleaned dataset

The index is reset to 0-based (row indices have gaps after row removals) and the cleaned dataframe is written to `../cleaned_dataset.csv`.

The exported file contains the **23 original columns** (after removing comment columns) plus **6 engineered features** added in steps 6–10:

| Feature | Type | Description |
|---------|------|-------------|
| `year` | integer | Year extracted from `Date` |
| `month` | integer | Month extracted from `Date` |
| `season` | string | Meteorological season |
| `delay_category` | ordered category | Severity label based on average arrival delay |
| `cancellation_rate` | float | Cancelled / scheduled |
| `punctuality_rate` | float | 1 − (delayed at arrival / scheduled) |

In [ ]:
df = df.reset_index(drop=True)
df.to_csv("../cleaned_dataset.csv", index=False)
print(f"Exported: {len(df)} rows, {len(df.columns)} columns → ../cleaned_dataset.csv")
print(f"Columns: {df.columns.tolist()}")

### 13. Data quality audit — pipeline self-evaluation

This section measures the **quality of the cleaned dataset** across five dimensions. All metrics are appended to the `report` accumulator and exported to `cleaning_report.csv` at step 14.

| Sub-section | What it measures |
|-------------|-----------------|
| **13.1 Completeness** | Fraction of non-null cells, overall and per column |
| **13.2 Numeric distributions** | n, mean, std, quartiles, min, max for key numeric columns |
| **13.3 Outlier rates** | Fraction of values with \|z-score\| > 3, per column |
| **13.4 Validity checks** | Binary invariants that must hold after cleaning (0 = fail, 1 = pass) |
| **13.5 Pipeline effectiveness** | Total corrections made and algebraic recoveries performed |

> These metrics do **not** modify the data — they describe it. They serve as evidence that the cleaning pipeline produced a dataset that is ready for modelling.

#### 13.1 Completeness

The fraction of non-null cells in the cleaned dataset, reported both overall and broken down per column.

A completeness of 100% would mean every cell has a value. Anything below that reflects the **intentional nulls** documented in step 4.8 — columns where derivation was not possible and imputation was deliberately avoided.

The five least and most complete columns are printed to highlight where residual sparsity is concentrated.

In [ ]:
total_cells = df.shape[0] * df.shape[1]
null_cells = int(df.isnull().sum().sum())
completeness = round(1 - null_cells / total_cells, 4)

col_completeness = (1 - df.isnull().mean()).sort_values()

report.append(
    {
        "metric": "overall_completeness",
        "value": completeness,
        "category": "quality_score",
        "reason": "Non-null cells / total cells (1.0 = fully complete)",
    }
)
for col in df.columns:
    rate = round(float(1 - df[col].isnull().mean()), 4)
    report.append(
        {
            "metric": f"completeness__{col}",
            "value": rate,
            "category": "completeness_per_column",
            "reason": col,
        }
    )

print(
    f"Overall completeness : {completeness:.2%}  ({total_cells - null_cells:,} / {total_cells:,} cells non-null)"
)
print(f"\nLeast complete columns:")
for col, rate in col_completeness.head(5).items():
    bar = "░" * int((1 - rate) * 30)
    print(f"  {rate:>6.1%}  {bar}  {col}")
print(f"\nMost complete columns:")
for col, rate in col_completeness.tail(5).items():
    print(f"  {rate:>6.1%}  {col}")

#### 13.2 Numeric distribution statistics

Descriptive statistics (count, mean, std, quartiles, min, max) for the most important numeric columns. These verify that the cleaning pipeline did not distort natural distributions — for example, that correcting negative values did not shift the median, or that capping impossible counts did not collapse variance.

Key things to look for in the output:

| Signal | Possible interpretation |
|--------|------------------------|
| `mean` close to `median` | Roughly symmetric distribution |
| Large gap between `p75` and `max` | Heavy right tail or surviving outliers |
| `std = NaN` or `mean = inf` | `inf` values still present in the column — a bug |

In [ ]:
dist_cols = [
    "Number of scheduled trains",
    "Average delay of all trains at arrival",
    "Average delay of all trains at departure",
    "Average delay of late trains at arrival",
    "Average delay of late trains at departure",
    "cancellation_rate",
    "punctuality_rate",
    "Average journey time",
    "Pct delay due to external causes",
    "Pct delay due to infrastructure",
    "Pct delay due to traffic management",
    "Pct delay due to rolling stock",
]

stat_rows = []
for col in dist_cols:
    if col not in df.columns:
        continue
    s = df[col].dropna()
    if s.empty:
        continue
    row = {
        "column": col,
        "n": len(s),
        "mean": round(float(s.mean()), 3),
        "std": round(float(s.std()), 3),
        "min": round(float(s.min()), 3),
        "p25": round(float(s.quantile(0.25)), 3),
        "median": round(float(s.median()), 3),
        "p75": round(float(s.quantile(0.75)), 3),
        "max": round(float(s.max()), 3),
    }
    stat_rows.append(row)
    for stat, val in row.items():
        if stat == "column":
            continue
        report.append(
            {
                "metric": f"dist__{col}__{stat}",
                "value": val,
                "category": "numeric_distribution",
                "reason": col,
            }
        )

stats_df = pd.DataFrame(stat_rows).set_index("column")
print(stats_df.to_string())

#### 13.3 Outlier rates

For each numeric column, the **outlier rate** is the fraction of rows whose value lies more than 3 standard deviations from the mean (|z-score| > 3).

For a normally distributed variable, only ~0.3% of values should fall outside this range. Rates significantly above 1% signal columns with extreme values that survived cleaning and may need further treatment — log transformation, winsorising, or careful handling — before being used in a model.

The bar chart includes a **5% threshold line** as a practical reference. Columns crossing it warrant attention.

In [ ]:
outlier_cols = [
    c for c in df.select_dtypes(include="number").columns if c not in ("year", "month")
]
outlier_rows = []

for col in outlier_cols:
    s = df[col].dropna()
    if len(s) < 10 or s.std() == 0:
        continue
    z = (s - s.mean()) / s.std()
    n_out = int((z.abs() > 3).sum())
    rate = round(n_out / len(df), 4)
    outlier_rows.append((col, n_out, rate))
    report.append(
        {
            "metric": f"outlier_rate__{col}",
            "value": rate,
            "category": "outlier_analysis",
            "reason": f"{n_out} values |z|>3 out of {len(df)} rows",
        }
    )

outlier_rows.sort(key=lambda x: -x[2])
print(f'{"Column":<58} {"N outliers":>10}  {"Rate":>7}')
print("─" * 82)
for col, n, rate in outlier_rows:
    bar = "█" * min(int(rate * 300), 30)
    print(f"{col:<58} {n:>10,}  {rate:>6.2%}  {bar}")

In [ ]:
filtered = [(col, rate) for col, n, rate in outlier_rows if rate > 0]
if filtered:
    cols_labels = [c for c, r in filtered]
    rates = [r for c, r in filtered]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(cols_labels, rates)
    ax.axvline(0.05, color="red", linestyle="--", label="5% threshold")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.set_title("Outlier rate per column (|z-score| > 3)")
    ax.legend()
    plt.tight_layout()
    plt.show()

#### 13.4 Feature validity checks

Binary assertions on invariants that **must** hold after a correct cleaning pipeline. Each check outputs `1` (pass) or `0` (fail). A failing check identifies a residual data quality issue that the pipeline did not fully resolve.

| Check | What it tests |
|-------|--------------|
| `cancellation_rate_in_0_1` | Rate is between 0 and 1 — no negative or > 100% cancellation |
| `punctuality_rate_in_0_1` | Rate is between 0 and 1 |
| `no_negative_scheduled` | No negative scheduled train count remains |
| `hierarchy_15min_30min` | Count > 15 min ≥ count > 30 min, for every row |
| `hierarchy_30min_60min` | Count > 30 min ≥ count > 60 min, for every row |
| `pct_delay_sum_le_105` | Delay cause percentages sum to at most 105% (5% rounding tolerance) |
| `date_range_sane` | All dates fall between 2015 and 2026 |
| `service_only_2_values` | Only `NATIONAL` and `INTERNATIONAL` appear in the `Service` column |
| `no_fully_null_rows` | No row has every column null |

In [ ]:
checks = {}

checks["cancellation_rate_in_0_1"] = int(
    df["cancellation_rate"].dropna().between(0, 1).all()
)
checks["punctuality_rate_in_0_1"] = int(
    df["punctuality_rate"].dropna().between(0, 1).all()
)
checks["no_negative_scheduled"] = int(
    (df["Number of scheduled trains"].dropna() >= 0).all()
)

hier_pairs = [
    ("Number of trains delayed > 15min", "Number of trains delayed > 30min"),
    ("Number of trains delayed > 30min", "Number of trains delayed > 60min"),
]
for col_lo, col_hi in hier_pairs:
    both = df[[col_lo, col_hi]].dropna()
    key = f'hierarchy_{col_lo.split(">")[1].strip().replace(" ", "")}_{col_hi.split(">")[1].strip().replace(" ", "")}'
    checks[key] = int((both.iloc[:, 0] >= both.iloc[:, 1]).all())

pct_cols = [c for c in df.columns if c.startswith("Pct delay due to")]
if pct_cols:
    row_pct_sum = df[pct_cols].sum(axis=1, skipna=True)
    checks["pct_delay_sum_le_105"] = int((row_pct_sum.dropna() <= 105).all())

checks["date_range_sane"] = int(
    (df["Date"] >= "2015-01-01").all() and (df["Date"] <= "2026-12-31").all()
)
checks["service_only_2_values"] = int(
    df["Service"].dropna().isin(["NATIONAL", "INTERNATIONAL"]).all()
)
checks["no_fully_null_rows"] = int(df.isnull().all(axis=1).sum() == 0)

all_passed = sum(checks.values())
for k, v in checks.items():
    report.append(
        {
            "metric": f"validity__{k}",
            "value": v,
            "category": "validity_checks",
            "reason": "1=pass  0=fail",
        }
    )
report += [
    {
        "metric": "validity_checks_passed",
        "value": all_passed,
        "category": "validity_checks",
        "reason": f"{all_passed}/{len(checks)} checks passed",
    },
    {
        "metric": "validity_checks_total",
        "value": len(checks),
        "category": "validity_checks",
        "reason": "Total validity rules evaluated",
    },
]

print("Validity checks:")
for k, v in checks.items():
    print(f'  {"✓" if v else "✗"} {k}')
print(f"\n{all_passed}/{len(checks)} checks passed")

In [ ]:
names = list(checks.keys())
vals = list(checks.values())
colors = ["green" if v == 1 else "red" for v in vals]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names, vals, color=colors)
ax.set_xticks([0, 1])
ax.set_xticklabels(["FAIL", "PASS"])
ax.set_title("Validity checks — pipeline invariants")
plt.tight_layout()
plt.show()

#### 13.5 Pipeline effectiveness

Two composite metrics quantify the total work performed by the pipeline:

| Metric | What it counts |
|--------|---------------|
| **Total corrections** | Impossible values fixed: negatives + count overflows + hierarchy violations (steps 11, 11.1, 11.2) |
| **Total recovered cells** | Missing values derived algebraically from existing data (steps 4.6, 4.7) |

The **correction rate per cell** (corrections / total cells in the final dataset) measures the density of data errors in the raw file. A low rate confirms that issues were rare rather than systemic.

The **recovery vs. initial null** ratio shows what fraction of the original missing values were filled in without any imputation.

In [ ]:
total_corrections = sum(
    next(r["value"] for r in report if r["metric"] == m)
    for m in [
        "values_fixed_negative",
        "values_fixed_count_vs_scheduled",
        "values_fixed_delay_hierarchy",
    ]
)
total_recovered = sum(
    next(r["value"] for r in report if r["metric"] == m)
    for m in [
        "values_filled_dep_late",
        "values_filled_dep_all",
        "values_filled_arr_late",
        "values_filled_arr_all",
    ]
)
total_cells_final = df.shape[0] * df.shape[1]
correction_rate = round(total_corrections / total_cells_final, 6)
initial_null_lookup = next(
    r["value"] for r in report if r["metric"] == "initial_null_total"
)
recovery_vs_null = round(total_recovered / max(initial_null_lookup, 1), 4)

report += [
    {
        "metric": "total_corrections",
        "value": total_corrections,
        "category": "pipeline_effectiveness",
        "reason": "Negatives + count overflows + hierarchy violations corrected",
    },
    {
        "metric": "total_recovered_cells",
        "value": total_recovered,
        "category": "pipeline_effectiveness",
        "reason": "Cells recovered by algebraic derivation (all 4 delay columns)",
    },
    {
        "metric": "correction_rate_per_cell",
        "value": correction_rate,
        "category": "pipeline_effectiveness",
        "reason": "Corrections / total final cells — data noise density",
    },
    {
        "metric": "recovery_vs_initial_null",
        "value": recovery_vs_null,
        "category": "pipeline_effectiveness",
        "reason": "Recovered / initial null count — algebraic coverage",
    },
]
print(f"Total corrections : {total_corrections:,}")
print(f"Total recovered   : {total_recovered:,}")
print(f"Correction rate   : {correction_rate:.4%} of all cells")
print(
    f"Recovery vs nulls : {recovery_vs_null:.1%} of initial nulls recovered algebraically"
)

### 14. Export — cleaning audit report

The final dataset state (shape and null count) is appended to the report accumulator, which is then written to `../cleaning_report.csv`.

This file provides a **complete audit trail** of every transformation applied by the pipeline — one row per metric, with its value, category, and a plain-language reason. It can be used to:
- Compare cleaning runs across different versions of the dataset
- Demonstrate data quality to stakeholders or reviewers
- Independently reproduce and verify every cleaning decision

In [ ]:
remaining_null = int(df.isnull().sum().sum())
report += [
    {
        "metric": "final_rows",
        "value": len(df),
        "category": "final_state",
        "reason": "Rows after full cleaning pipeline",
    },
    {
        "metric": "final_columns",
        "value": len(df.columns),
        "category": "final_state",
        "reason": "Columns after cleaning + feature engineering",
    },
    {
        "metric": "final_null_total",
        "value": remaining_null,
        "category": "final_state",
        "reason": "Intentionally kept null — non-derivable columns",
    },
    {
        "metric": "total_rows_removed",
        "value": original_file.shape[0] - len(df),
        "category": "final_state",
        "reason": "Total rows removed across all cleaning steps",
    },
]

report_df = pd.DataFrame(report, columns=["metric", "value", "category", "reason"])
report_df.to_csv("../cleaning_report.csv", index=False)

print(f"Report saved: {len(report_df)} entries → ../cleaning_report.csv")
print(report_df["category"].value_counts().to_string())

### 15. Exploratory Data Analysis

The dataset is now clean and ready for analysis. This section explores the key patterns in TGV punctuality through a series of targeted visualisations — no data is modified here.

| Plot | Question it answers |
|------|---------------------|
| **B1** — Delay distribution | How are arrival delays distributed? Symmetric or skewed? |
| **B2** — Seasonal boxplot | Does the season affect delay levels and variance? |
| **B3** — Top stations by delay | Which departure hubs produce the highest average delays? |
| **B4** — Monthly trend | How has punctuality evolved over time? Are there visible crisis periods? |
| **B5** — Delay category breakdown | What share of services fall into each severity category? |
| **B6** — Correlation heatmap | Which features are most strongly associated with arrival delay? |
| **B7** — Cancellation by service | Does service type (NATIONAL vs. INTERNATIONAL) affect cancellation rates? |

**B1 — Distribution of average arrival delay**

A histogram with a kernel density estimate (KDE) overlay, alongside a box plot.

The KDE gives a smooth view of the overall shape; the box plot highlights the median, interquartile range, and outliers. Together they reveal whether the delay distribution is symmetric, right-skewed, or multimodal — and whether extreme outliers dominate the tail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    df["Average delay of all trains at arrival"].dropna(), kde=True, ax=axes[0]
)
axes[0].set_title("Distribution of average arrival delay")
axes[0].set_xlabel("Delay (minutes)")
sns.boxplot(y=df["Average delay of all trains at arrival"].dropna(), ax=axes[1])
axes[1].set_title("Arrival delay — spread and outliers")
plt.tight_layout()
plt.show()

**B2 — Arrival delay by season**

A box plot grouping all services by the season in which they operated.

If winter shows higher medians and wider spread, that confirms a weather effect. If summer has elevated medians despite lower variance, it points to load-induced congestion rather than disruption events. Comparing the four seasons side by side makes these patterns immediately visible.

In [ ]:
sns.boxplot(
    x="season",
    y="Average delay of all trains at arrival",
    data=df,
    order=["winter", "spring", "summer", "autumn"],
)
plt.title("Arrival delay by season")
plt.xlabel("Season")
plt.ylabel("Delay (minutes)")
plt.tight_layout()
plt.show()

**B3 — Top 10 departure stations by average arrival delay**

A horizontal bar chart showing the 10 departure stations with the highest mean arrival delay, averaged across all routes and months.

High-delay hubs are typically high-traffic interchange stations where platform congestion or scheduling constraints propagate downstream through the network. Identifying them is a first step toward targeted improvements.

In [ ]:
top_stations = (
    df.groupby("Departure station")["Average delay of all trains at arrival"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
fig, ax = plt.subplots(figsize=(10, 6))
top_stations.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 10 departure stations by average arrival delay")
ax.set_xlabel("Average delay (minutes)")
plt.tight_layout()
plt.show()

**B4 — Monthly average arrival delay over time**

A line chart of the mean arrival delay per calendar month, from the first record to the most recent.

The dashed red line marks the overall mean across the entire dataset. Sustained periods above it signal crisis episodes — major strikes, infrastructure incidents, or the COVID-19 disruptions visible around 2020. Periods below it indicate improved operational performance.

In [ ]:
monthly = df.groupby(df["Date"].dt.to_period("M"))[
    "Average delay of all trains at arrival"
].mean()
monthly.index = monthly.index.to_timestamp()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly.values)
ax.axhline(monthly.mean(), color="red", linestyle="--", label="overall mean")
ax.set_title("Monthly average arrival delay over time")
ax.set_xlabel("Date")
ax.set_ylabel("Delay (minutes)")
ax.legend()
plt.tight_layout()
plt.show()

**B5 — Delay severity category breakdown**

A pie chart and a bar chart showing the distribution of services across the five delay categories defined in step 8.

The pie gives an immediate sense of proportions; the bar chart makes it easier to compare exact counts. `severe` delays (> 30 min) are rare but represent the most impactful disruptions for passengers and are the threshold at which compensation is triggered under EU rail regulation.

In [ ]:
delay_counts = df["delay_category"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(delay_counts.values, labels=delay_counts.index, autopct="%1.1f%%")
axes[0].set_title("Delay category proportions")
delay_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title("Delay category counts")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Number of services")
plt.tight_layout()
plt.show()

**B6 — Correlation matrix**

A heatmap of Pearson correlations between key numeric features, using a diverging colour scale (blue = negative correlation, red = positive correlation).

Strong positive correlations between delay columns confirm that disruptions cascade: a departure delay almost always becomes an arrival delay. The strong negative correlation between arrival delay and punctuality rate is expected by definition. The heatmap also reveals which delay cause percentages are most associated with overall delay — useful for identifying the main drivers of poor punctuality.

In [ ]:
heatmap_cols = [
    "Average delay of all trains at arrival",
    "Average delay of all trains at departure",
    "Average delay of late trains at arrival",
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
    "cancellation_rate",
    "punctuality_rate",
    "Average journey time",
]
heatmap_cols = [c for c in heatmap_cols if c in df.columns]
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    df[heatmap_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Correlation matrix — key numeric features")
plt.tight_layout()
plt.show()

**B7 — Cancellation rate by service type**

A box plot comparing the distribution of `cancellation_rate` between `NATIONAL` and `INTERNATIONAL` services.

International routes involve cross-border infrastructure and multi-operator scheduling, introducing additional failure points compared to domestic routes. If the `INTERNATIONAL` box is shifted higher or wider, it confirms that cross-border complexity translates into higher cancellation risk.

In [ ]:
sns.boxplot(x="Service", y="cancellation_rate", data=df)
plt.title("Cancellation rate by service type")
plt.ylabel("Cancellation rate")
plt.tight_layout()
plt.show()